# CompoCLIP role-3 evaluation on Colab Enterprise (GCS)

This notebook is for a **Colab Enterprise runtime** in project
`cs5489-gp-121`. It does **not** use `google.colab.drive` or `/content`.

## Before you run

1. Upload these files to one GCS prefix, for example
   `gs://YOUR_BUCKET/compoclip/`:

   ```text
   A2_eval.tar      2.67 GB
   A2_models.tar    3.03 GB
   data_role3.tar   0.55 GB
   ```

2. Give the runtime service account read access to that prefix and
   write access to the results prefix. `roles/storage.objectAdmin` on
   the bucket is the simplest choice for a private course bucket.

3. Create/select a Colab Enterprise runtime with a **T4 or L4 GPU** and
   at least **50 GB disk**. The default 100 GB boot disk is enough for
   this notebook.

The repository is cloned from GitHub, so the code (including
`A2_eval_tools/`) comes from the pushed `main` branch, not from this
notebook upload.

## 1. Configuration

In [ ]:
import os
from pathlib import Path

# Either edit this line or set COMPOCLIP_BUCKET before starting the kernel.
BUCKET_URI = os.environ.get("COMPOCLIP_BUCKET", "gs://REPLACE_ME/compoclip")
REPO_URL = os.environ.get(
    "COMPOCLIP_REPO", "https://github.com/wh-violet667/CompoCLIP.git"
)

assert BUCKET_URI.startswith("gs://"), "BUCKET_URI must start with gs://"
assert "REPLACE_ME" not in BUCKET_URI, (
    "Set COMPOCLIP_BUCKET (e.g. gs://my-bucket/compoclip) or edit BUCKET_URI."
)

WORK = Path("/tmp/compoclip")
REPO = WORK / "CompoCLIP"
DATA_ROOT = WORK / "data_root"
DOWNLOADS = WORK / "downloads"
for path in (WORK, DATA_ROOT, DOWNLOADS):
    path.mkdir(parents=True, exist_ok=True)

print("bucket :", BUCKET_URI)
print("repo   :", REPO_URL)
print("work   :", WORK)

## 2. Check GPU and disk

In [ ]:
!nvidia-smi

In [ ]:
import shutil
import torch

free_gb = shutil.disk_usage("/tmp").free / 1e9
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
print("free disk under /tmp: %.1f GB" % free_gb)
assert torch.cuda.is_available(), "Select a GPU runtime (T4/L4), not a CPU runtime."
print("GPU:", torch.cuda.get_device_name(0))
assert free_gb > 20, "Need at least ~20 GB free under /tmp for data + results."

## 3. Clone the repository

In [ ]:
import subprocess

if (REPO / ".git").exists():
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO)])
print(subprocess.check_output(["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True))

## 4. Install evaluation dependencies

Colab Enterprise GPU images usually already provide CUDA-enabled
`torch`/`torchvision`. We install `open_clip_torch` with `--no-deps` so
pip cannot replace the matching torch wheel.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchvision") is None:
    raise RuntimeError(
        "This runtime has no torch/torchvision. Choose a Colab Enterprise GPU "
        "runtime image that includes PyTorch, or install the CUDA wheel that "
        "matches the runtime driver before continuing."
    )

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "open_clip_torch==2.24.0"]
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q",
     "timm", "ftfy", "regex", "huggingface_hub", "safetensors",
     "tqdm", "pandas", "pyyaml", "google-cloud-storage"]
)
import open_clip, timm
print("open_clip:", open_clip.__file__)
print("timm:", timm.__version__)

## 5. Download the three packages from GCS

In [ ]:
from urllib.parse import urlparse

from google.cloud import storage

parsed = urlparse(BUCKET_URI)
bucket_name = parsed.netloc
prefix = parsed.path.strip("/")
client = storage.Client()
bucket = client.bucket(bucket_name)

for name in ("A2_eval.tar", "A2_models.tar", "data_role3.tar"):
    key = f"{prefix}/{name}" if prefix else name
    target = DOWNLOADS / name
    blob = bucket.get_blob(key)
    if blob is None:
        raise FileNotFoundError(f"gs://{bucket_name}/{key} does not exist")
    if target.exists() and target.stat().st_size == blob.size:
        print("cached", target, f"({target.stat().st_size / 1e9:.2f} GB)")
        continue
    print(f"downloading gs://{bucket_name}/{key} -> {target}", flush=True)
    blob.download_to_filename(str(target))
    print(f"  done {target.stat().st_size / 1e9:.2f} GB", flush=True)

## 6. Extract and link the data

In [ ]:
import shutil
import tarfile

for name in ("A2_eval.tar", "A2_models.tar", "data_role3.tar"):
    print("extracting", name, flush=True)
    with tarfile.open(DOWNLOADS / name) as tar:
        try:
            tar.extractall(DATA_ROOT, filter="data")
        except TypeError:  # Python < 3.12
            tar.extractall(DATA_ROOT)
    print("  ok", flush=True)

def replace_with_symlink(source: Path, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_symlink() or target.is_file():
        target.unlink()
    elif target.exists():
        shutil.rmtree(target)
    target.symlink_to(source, target_is_directory=True)
    print("linked", target, "->", source)

replace_with_symlink(DATA_ROOT / "A2_eval", REPO / "A2_eval")
replace_with_symlink(DATA_ROOT / "A2_models", REPO / "A2_models")
for subset in ("golden", "flickr30k", "winoground"):
    replace_with_symlink(DATA_ROOT / "data" / subset, REPO / "data" / subset)

assert (REPO / "A2_eval_tools" / "evaluate_models.py").is_file(), (
    "A2_eval_tools missing from the repo clone; push it or set COMPOCLIP_REPO."
)
print("data layout ready")

## 7. Rewrite the Windows absolute paths

`A2_eval/data/*.jsonl` and `data/winoground/pairs.jsonl` were prepared on
Windows. Both must be rewritten before evaluation, otherwise every image
path fails to open on Linux.

In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "scripts/make_a2_eval_portable.py",
        "--root", str(REPO / "A2_eval"),
        "--data-root", str(REPO / "data"),
    ],
    cwd=REPO,
)

In [ ]:
import json
from pathlib import Path

pairs = Path(REPO / "data" / "winoground" / "pairs.jsonl")
rows = [json.loads(line) for line in pairs.read_text(encoding="utf-8").splitlines() if line.strip()]
image_root = Path(REPO / "data" / "winoground" / "images")
for row in rows:
    for key in ("image_0", "image_1"):
        row[key] = str(image_root / Path(str(row[key]).replace("\\", "/")).name)
pairs.write_text(
    "\n".join(json.dumps(row, ensure_ascii=False) for row in rows) + "\n",
    encoding="utf-8",
)
print("winoground paths fixed:", len(rows))

## 8. E3/E4b: A2 4/3 protocol (ARO + SugarCrepe + COCO)

This is the long GPU job (roughly tens of minutes on a T4). Results go to
`A2_eval/results/<tag>/evaluation.json` and per-sample `.npz` files.

In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "scripts/evaluate_role3_models.py",
        "--tags", "e3", "e4b",
        "--benchmarks", "aro", "sugarcrepe", "coco",
        "--device", "cuda",
        "--image-batch", "64",
        "--text-batch", "256",
    ],
    cwd=REPO,
)

## 9. Flickr30k + Winoground retrieval (E1/E2/E2a/E3/E4b)

E0 is already archived and E4a = E2, so neither is rerun. Each run writes
a row to `evals-mm/results/<benchmark>.csv`.

In [ ]:
import subprocess
import sys

tags = ["e1", "e2", "e2a", "e3", "e4b"]
for tag in tags:
    checkpoint = REPO / "A2_models" / tag / f"{tag}_model.pt"
    assert checkpoint.is_file(), checkpoint
    for benchmark in ("flickr30k", "winoground"):
        cmd = [
            sys.executable,
            "evals-mm/run_eval.py",
            "--tag", tag,
            "--checkpoint", str(checkpoint),
            "--benchmark", benchmark,
        ]
        print("\n$", " ".join(cmd), flush=True)
        subprocess.check_call(cmd, cwd=REPO)

## 10. Rebuild the result tables

In [ ]:
import subprocess
import sys

for script in (
    "scripts/assemble_role3_results.py",
    "scripts/build_full_master.py",
    "scripts/make_master_md.py",
):
    print("\n$", script, flush=True)
    subprocess.check_call([sys.executable, script], cwd=REPO)

## 11. Upload the results to GCS

`/tmp` disappears when the runtime is deleted, so copy the artifacts
before shutting it down. Relative paths are preserved.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse

from google.cloud import storage

parsed = urlparse(BUCKET_URI)
prefix = parsed.path.strip("/")
out_prefix = f"{prefix}/results_role3" if prefix else "results_role3"
client = storage.Client()
bucket = client.bucket(parsed.netloc)

sources = [
    (REPO / "report" / "results", "report_results"),
    (REPO / "evals-mm" / "results", "evals_mm_results"),
    (REPO / "A2_eval" / "results" / "e3", "a2_eval_e3"),
    (REPO / "A2_eval" / "results" / "e4b", "a2_eval_e4b"),
]
uploaded = 0
for source, label in sources:
    if not source.exists():
        print("skip missing", source)
        continue
    for path in source.rglob("*"):
        if not path.is_file() or path.suffix in {".pyc"}:
            continue
        relative = path.relative_to(source).as_posix()
        blob = bucket.blob(f"{out_prefix}/{label}/{relative}")
        blob.upload_from_filename(str(path))
        uploaded += 1
print(f"uploaded {uploaded} files to gs://{parsed.netloc}/{out_prefix}")

## Notes and failure recovery

- **Runtime has no internet:** upload a repository zip to the same GCS
  prefix and change the clone cell to download and extract it.
- **Permission denied on GCS:** grant the runtime service account
  `roles/storage.objectAdmin` on the bucket, or at least
  `objectViewer` + `objectCreator` on the results prefix.
- **A2 4/3 job dies halfway:** re-run cell 8. Feature caches under
  `A2_eval/cache/` make a re-run resume from the last completed
  benchmark. Do not change `--image-batch`/`--text-batch` between runs,
  or the cache identity changes.
- **Retrieval job dies:** re-run cell 9 with a shorter `tags` list.
  Existing CSV rows are kept and rewritten deterministically.
- **Do not fill missing report numbers by hand.** If a job fails, leave
  the cell marked incomplete and bring the error back.